# Debug ConfigManager Issue

Troubleshooting `Missing database config key: user` error in `ConfigManager`.
Testing loading of `config/demo_config.yaml` and `DATABASE_URL` parsing.

In [1]:
import sys
import os

# Add project root to Python path
project_root = os.path.abspath(os.getcwd())
sys.path.append(project_root)
print(f"Project root added to path: {project_root}")
print(f"sys.path: {sys.path}")

# Verify helper directory
print("Contents of helper/: ", os.listdir('helper'))

Project root added to path: P:\DynaPOD\proj\trader
sys.path: ['C:\\pyver\\py312\\python312.zip', 'C:\\pyver\\py312\\DLLs', 'C:\\pyver\\py312\\Lib', 'C:\\pyver\\py312', 'P:\\DynaPOD\\proj\\trader\\venv', '', 'P:\\DynaPOD\\proj\\trader\\venv\\Lib\\site-packages', 'P:\\DynaPOD\\proj\\trader\\venv\\Lib\\site-packages\\win32', 'P:\\DynaPOD\\proj\\trader\\venv\\Lib\\site-packages\\win32\\lib', 'P:\\DynaPOD\\proj\\trader\\venv\\Lib\\site-packages\\Pythonwin', 'P:\\DynaPOD\\proj\\trader']
Contents of helper/:  ['config_manager.py', 'database.py', 'Logger.py', 'md_config_manager.md', 'timelimit.py', 'timer.py', 'timezone.py', 'xml_config_manager.py', '__init__.py', '__pycache__']


## Load ConfigManager and Inspect config/demo_config.yaml

Load the config file and check its contents.

In [6]:
from helper.config_manager import ConfigManager

# Check if config/demo_config.yaml exists
config_file = 'config/demo_config.yaml'
print(f"Config file exists: {os.path.exists(config_file)}")
if os.path.exists(config_file):
    with open(config_file, 'r') as f:
        print("Raw config file contents:\n", f.read())

# Load config
try:
    config = ConfigManager(config_file)
    print("ConfigManager loaded successfully")
except Exception as e:
    print(f"Failed to load ConfigManager: {e}")

Config file exists: True
Raw config file contents:
 logging:
  logger_name: "TradingLogger"
  log_stream_level: "INFO"
  logstream_format: "%(name)s - %(levelname)s - %(message)s"
  log_dir_name: "logs"
  log_file_max_bytes: 10485760  # 10MB
  log_file_backup_count: 10
  logfile_file_format: "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

timelimit:
  max_run_time_seconds: 3600  # 1 hour

timezone:
  name: "America/Chicago"

# Database settings (used locally, overridden by DATABASE_URL on Heroku)
database:
  user: "postgres"
  password: "your_password"
  host: "localhost"
  port: "5432"
  database: "test_db"
ConfigManager loaded successfully


## Test ConfigManager Get Methods

Check if `get` and `get_with_default` return expected values.

In [3]:
# Test without DATABASE_URL
print("Testing without DATABASE_URL...")
if 'DATABASE_URL' in os.environ:
    del os.environ['DATABASE_URL']

print("Logging.logger_name:", config.get('logging', 'logger_name'))
print("Database.user:", config.get('database', 'user'))
print("Database.user with default:", config.get_with_default('database', 'user', default='default_user'))

Testing without DATABASE_URL...
Logging.logger_name: TradingLogger
Database.user: postgres
Database.user with default: default_user


## Test with DATABASE_URL

Set `DATABASE_URL` and test again.

In [4]:
os.environ['DATABASE_URL'] = 'postgres://u64g8eocr7nhul:p3b8ecfc9a637f11fb55f26fcf499d36f5f6a5c90a59f1c99a40fce32e4b549b7@ca932070ke6bv1.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com:5432/da1co8rkl4afh'
print("DATABASE_URL set:", os.environ.get('DATABASE_URL'))

# Reload ConfigManager to ensure it picks up the env var
config = ConfigManager('config/demo_config.yaml')

print("Database.user (should use DATABASE_URL):", config.get('database', 'user'))
print("Database.password:", config.get('database', 'password'))
print("Database.host:", config.get('database', 'host'))
print("Database.port:", config.get('database', 'port'))
print("Database.database:", config.get('database', 'database'))

DATABASE_URL set: postgres://u64g8eocr7nhul:p3b8ecfc9a637f11fb55f26fcf499d36f5f6a5c90a59f1c99a40fce32e4b549b7@ca932070ke6bv1.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com:5432/da1co8rkl4afh
Database.user (should use DATABASE_URL): postgres
Database.password: your_password
Database.host: localhost
Database.port: 5432
Database.database: test_db


## Test DatabaseHandler

Attempt to initialize `DatabaseHandler` with the config.

In [5]:
from helper.Logger import Logging
from helper.database import DatabaseHandler

logger = Logging(config, instance_id="debug_config")
try:
    db_handler = DatabaseHandler(config, logger)
    engine = db_handler.get_engine()
    print("DatabaseHandler initialized successfully")
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Connection test result:", result.fetchone())
except Exception as e:
    print(f"Failed to initialize DatabaseHandler: {e}")

MyLogger.debug_config - ERROR - Error creating PostgreSQL engine: Missing database config key: user


Failed to initialize DatabaseHandler: Missing database config key: user
